# ECM Neighborhood Analysis v4

ECM neighborhoods of the Visium cohort (Spencer Watson; moved from `analysis/survival/ECM_niches_v4.ipynb`).

Produces:
- **Supp Table 3a**: the brain/GBM matrisome panel (117 genes) and its expression summary.
- ECM clustering and kNN neighborhoods (`ecm_neighborhood_annotations_v4.csv`, used by every
  ECM analysis of the paper); off by default (`RUN_CLUSTERING = False` reads the frozen file).
- **Fig 4c**: ECM neighborhoods on the H&E of CH_G_196a.
- **Fig 4d**: z-scores of the mean matrisome expression per neighborhood, the 40 "most variable"
  genes (the published panel; the legend's "top 5" does not describe it).
- ECM neighborhood gene signatures, matrisome panel and full transcriptome (Methods).
- **Supp Fig 4f**: cell-type composition per ECM neighborhood.
- **Fig 4g**: spatial contact frequency between neighborhoods, with permutation p values.
- ECM neighborhood abundance vs overall survival (Spearman).

Other ECM panels: Fig 4e and Supp Fig 4e, Supp Tables 3b–c in `13_colocalization_plots.ipynb`;
Fig 4f in `14_pathologist_annotation_analyses.py`; Fig 4h in `15_neighborhood_survival.ipynb`;
the TIL-zone cells in `18_ECM_supplement.ipynb`.

Changes from the original notebook:
- Loading cells (2–12) replaced by the package (`gbmcollaboration.paper.neighborhoods` is imported as `nbhd`: the notebook uses `neighborhoods` as a variable): `nbhd.load_spots` (spots and
  coordinates, loader exclusions of `load_spatial_adata`), `visium.load_expression` with the
  matrisome genes only (log_norm_count, cast to float64 as `load_spatial_adata` returned it; the
  full dense matrix is > 100 GB), `visium.load_c2l` / `add_c2l_to_obs` (cell2location means),
  `annotation.add_celltype_key(efs_copy=True)` (the EFS key the notebook read),
  `cohort.clinical_as_ran()` (cell 4). Spots are in the order of the frozen annotation file
  (the order of `load_spatial_adata`), which the permutations of the adjacency test depend on.
- Cell 14 split: the gene list comes before loading; the Supp Table 3a gene table (with the
  category of each gene from the comments of the list) is written here. The published table
  also lists the 6 genes commented out of the list (printed).
- Clustering cells 16, 18, 24, 40 run only with `RUN_CLUSTERING = True` and write to `OUT`
  instead of `Analysis_Repo/Notebooks`; otherwise cell 42 reads `config.ECM_ANNOTATIONS_CSV`.
  Cell 20 (reload of the Leiden clusters) removed.
- Fig 4c (cell 64): `sample_id` set to the published section CH_G_196a_vis (last run:
  ER_G_666a_vis).
- Cell 46 (full transcriptome): pseudobulk means from `neighborhoods.pseudobulk_sums` (per
  section × neighborhood sums, read a few sections at a time) instead of the full matrix.
- Cell 46 tested `imagerow` and `imagecol` as genes (they were columns of the expression matrix of
  the last run); they are not genes and are left out here, so BH FDR is over 18,082 instead of
  18,084 tests (max change 1e-4; no signature gene changes).
- Cell 30: the cell2location → curated-type block is `neighborhoods.c2l_curated_counts`.
- Cell 58: `compute_contacts` counts edges with `neighborhoods.contact_counts` (`np.add.at`)
  instead of a Python loop over 2.5 M edges × 1000 permutations; same labels shuffles, same counts.
- Figures and tables go to `OUT` (`output.save` / `output.save_table`) instead of the working
  directory; paper panels also to the panels folder. Comparisons with the outputs of the last run
  (`Analysis_Repo/Notebooks`, `Final_Notebooks`) are printed.
- Per-cell imports moved to the top; versions cell (1) → `output.versions`.
- Removed: 1 (versions), 2–12 (loading, see above), 20, 22 (UMAP), 26 (one-sample plot),
  28 (per-sample barplot; Supp Fig 4e is drawn in 13_colocalization_plots), 34–38 (survival,
  now `15_neighborhood_survival.ipynb`), 49 (composition-marker heatmap, not in the paper),
  51–54 and 66 (TIL zones, now `18_ECM_supplement.ipynb`), 60 (summary table, not a paper
  table), 62–63 (sample lookups).

In [ ]:
import gc
import json
import os
import tempfile
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import openslide
import pandas as pd
import s3fs
import scanpy as sc
import squidpy as sq
from matplotlib.collections import PatchCollection
from matplotlib.colors import to_hex, to_rgb
from matplotlib.lines import Line2D
from matplotlib.patches import Circle
from pandas.api.types import CategoricalDtype
from scipy import sparse
from scipy.cluster.hierarchy import dendrogram, leaves_list, linkage
from scipy.sparse import issparse
from scipy.stats import ranksums, spearmanr
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from statsmodels.stats.multitest import multipletests

from gbmcollaboration.paper import annotation, cohort, config, output, visium
from gbmcollaboration.paper import neighborhoods as nbhd

In [ ]:
# Parameters
OUT = output.setup("11_ECM_niches_v4")
sc.set_figure_params(dpi=150, color_map="viridis_r")
sc.settings.verbosity = 0

# True: re-run the ECM clustering and kNN neighborhoods (writes the annotation file to OUT);
# False: read the frozen annotations that every analysis of the paper used
RUN_CLUSTERING = False
ECM_ANNOTATIONS_PATH = config.ECM_ANNOTATIONS_CSV
LAYER = "log_norm_count"

# Outputs of the notebook's last run, to compare with
PUBLISHED_NB_DIR = config.EFS / "Analysis_Repo/Notebooks"
PUBLISHED_TABLES = config.EFS / "supplementary_tables"
# Fig 4d: the 40 genes of the published panel (= Final_Notebooks/ECM_neighborhood_heatmap_top40.png), left to right
PUBLISHED_FIG4D_GENES = [
    "PTPRZ1", "CSPG5", "BCAN", "NCAN", "LAMB1", "COL1A2", "COL5A2", "COL3A1", "COL1A1", "DCN",
    "COL18A1", "AGRN", "MMP2", "COL4A1", "COL4A2", "NID1", "BGN", "LAMA4", "TNC", "LGALS1",
    "CTSB", "TIMP1", "PLOD1", "COL6A1", "EFEMP1", "GPC1", "VCAN", "LAMB2", "MMP14", "LAMC1",
    "TGFBI", "CCN1", "FN1", "COL6A2", "COL5A1", "IGFBP5", "PLOD2", "P4HA1", "SERPINE1", "CHI3L1",
]

### Define ECM genes in matrisome list

In [ ]:
# ### Define brain/GBM ECM gene panel and assess expression/heterogeneity


# Core ECM + remodelers + matricellular panel for brain/GBM
ecm_genes_brain = [
    # Collagens
    "COL1A1", "COL1A2", "COL3A1", "COL4A1", "COL4A2",
    "COL5A1", "COL5A2", "COL5A3", "COL6A1", "COL6A2", "COL6A3",
    "COL7A1", "COL8A1", "COL11A1", "COL12A1",
    "COL14A1", "COL15A1", "COL18A1",
    "COL10A1", # "COL16A1", 

    # ECM glycoproteins
    "FN1", "FBN1", "FBN2",
    "TNC", "TNR", "TNXB",
    "SPARC", "POSTN", "SPP1", "EFEMP1",
    "LAMA1", "LAMA2", "LAMA4", "LAMA5",
    "LAMB1", "LAMB2", "LAMB3", 
    "LAMC1", "LAMC3",
    "THBS1", "THBS2", "THBS4",
    "VTN", 
    "NID1", "NID2",
    "FBLN1", "FBLN2", "FBLN5",
    "MMRN2", "VWA1", "TGFBI", "LTBP1",
    "CHI3L1", # "IGFBP7",
    "LGALS1", # "LGALS3",
    "CCN1", "CCN2",
    "EMILIN1",

    # Proteoglycans
    "VCAN", "ACAN", "BCAN", "NCAN",
    "DCN", "BGN", "LUM", "PRELP", "FMOD", "ASPN",
    "HSPG2", "AGRN",
    "GPC1", "GPC2", "GPC3",
    "SDC1", "SDC4",
    "CSPG4", "CSPG5", "PTPRZ1",
    "HAPLN1", "HAPLN2", "HAPLN3", "HAPLN4",

    # Proteases
    "MMP2", "MMP7", "MMP9", "MMP11","MMP12", "MMP13", "MMP14", "MMP16",
    "ADAMTS1", "ADAMTS4", "ADAMTS5", 
    "ADAM8",
    "HPSE",
    "PLAU", "PLAT",
    "CTSB", # "CTSD",

    # Protease inhibitors
    "TIMP1", "TIMP2", "TIMP3",
    "SERPINE1",

    # Crosslinking / collagen maturation enzymes
    "LOX", "LOXL1", "LOXL2", "LOXL3", "LOXL4",  # restored (4 patients)
    "TGM2",
    "P4HA1", "P4HA2",
    "PLOD1", "PLOD2", # "PLOD3",
    "BMP1", "PCOLCE",

    # Hyaluronan metabolism
    "HAS2", "HAS3",
    "HYAL2",

    # Optional inclusions
    # "IGFBP2", 
    "IGFBP5",  # GBM-associated secreted ECM factors
    "SRGN",    # secreted proteoglycan, myeloid/mast ECM
]

# Category of each gene, from the comments of the list above (Supp Table 3a)
_starts = {"COL1A1": "Collagens", "FN1": "ECM glycoproteins", "VCAN": "Proteoglycans", "MMP2": "Proteases",
           "TIMP1": "Protease inhibitors", "LOX": "Crosslinking / collagen maturation enzymes",
           "HAS2": "Hyaluronan metabolism", "IGFBP5": "Other"}
ecm_gene_category, _cat = {}, None
for g in ecm_genes_brain:
    _cat = _starts.get(g, _cat)
    ecm_gene_category[g] = _cat

### Load Visium data

Spots, coordinates and cell2location abundances of the Visium cohort with the exclusions of
`load_spatial_adata`, and the log-normalised expression of the matrisome genes only (the full
dense matrix is > 100 GB).

In [ ]:
obs, _ = nbhd.load_spots()
adata = visium.load_expression(genes=ecm_genes_brain, normalization=LAYER)[obs.index].copy()
adata.layers[LAYER] = adata.layers[LAYER].astype(np.float64)   # load_spatial_adata returned float64
adata.obs[["imagerow", "imagecol"]] = obs[["imagerow", "imagecol"]]

# Cell2Location mean abundances, merged into adata.obs by (sample, barcode)
visium.add_c2l_to_obs(adata, visium.load_c2l())

# Cell-type annotation key (EFS copy, as the notebook read it)
annotation.add_celltype_key(adata, efs_copy=True)
print(adata)

### Supp Table 3a: matrisome panel and its expression

In [ ]:
# Use the normalization layer if present, otherwise fallback to X
if hasattr(adata, "layers") and "log_norm_count" in adata.layers:
    X_source = adata.layers["log_norm_count"]
else:
    X_source = adata.X

if issparse(X_source):
    X_source = X_source.tocsr()

var_names = adata.var_names

# Determine which genes are present in the AnnData
ecm_genes_unique = sorted(set(ecm_genes_brain))
present_genes = [g for g in ecm_genes_unique if g in var_names]
missing_genes = [g for g in ecm_genes_unique if g not in var_names]

print(f"Total ECM genes in panel: {len(ecm_genes_unique)}")
print(f"Present in adata: {len(present_genes)}")
print(f"Missing from adata: {len(missing_genes)}")

if missing_genes:
    print("Example missing genes:", missing_genes[:10])

# Extract submatrix for present genes (spots x genes)
X_sub = adata[:, present_genes].X if X_source is adata.X else X_source[:, [var_names.get_loc(g) for g in present_genes]]

if issparse(X_sub):
    X_sub = X_sub.toarray()

n_spots = adata.n_obs
samples = adata.obs["sample"].to_numpy()
unique_samples, sample_inverse = np.unique(samples, return_inverse=True)
n_samples = len(unique_samples)
spots_per_sample = np.bincount(sample_inverse)

# Boolean expression mask (non-zero in the chosen layer)
expr_mask = X_sub > 0

metrics = []

for j, gene in enumerate(present_genes):
    vals = X_sub[:, j]
    expr = expr_mask[:, j]

    n_expr = int(expr.sum())
    pct_expr = (n_expr / n_spots) * 100.0

    mean_all = float(vals.mean())
    std_all = float(vals.std())
    mean_nonzero = float(vals[expr].mean()) if n_expr > 0 else 0.0
    cv_all = std_all / (mean_all + 1e-8)

    # Per-sample expression statistics
    expr_per_sample = np.bincount(sample_inverse, weights=expr.astype(np.int64))
    mean_expr_per_sample = np.bincount(sample_inverse, weights=vals) / (spots_per_sample + 1e-8)
    frac_expr_per_sample = expr_per_sample / (spots_per_sample + 1e-8)

    n_samples_ge_5pct = int((frac_expr_per_sample >= 0.05).sum())
    mean_frac_expr_per_sample = float(frac_expr_per_sample.mean())
    cv_sample_mean_expr = float(mean_expr_per_sample.std() / (mean_expr_per_sample.mean() + 1e-8))

    metrics.append(
        {
            "gene": gene,
            "n_spots_expr": n_expr,
            "pct_spots_expr": pct_expr,
            "mean_expr_all_spots": mean_all,
            "mean_expr_nonzero_spots": mean_nonzero,
            "cv_expr_all_spots": cv_all,
            "n_samples_ge_5pct_spots_expr": n_samples_ge_5pct,
            "n_samples_total": n_samples,
            "mean_frac_spots_expr_per_sample": mean_frac_expr_per_sample,
            "cv_mean_expr_per_sample": cv_sample_mean_expr,
        }
    )

ecm_expr_summary = (
    pd.DataFrame(metrics)
    .set_index("gene")
    .sort_values("pct_spots_expr", ascending=False)
)

print("\nECM expression summary (head):")
print(ecm_expr_summary.head(20))

supp3a = pd.DataFrame({"source": [ecm_gene_category[g] for g in ecm_genes_brain], "target": ecm_genes_brain})
output.save_table(OUT, "ECM_genes", supp3a, panel="Supp Table 3a", index=False)
output.save_table(OUT, "ECM_expression_summary", ecm_expr_summary)

published_3a = pd.read_csv(PUBLISHED_TABLES / "4a_ECM_genes.csv")
print(f"\nPublished gene table: {len(published_3a)} genes; here {len(supp3a)}")
print("  in published only:", sorted(set(published_3a["target"]) - set(supp3a["target"])))
print("  here only:", sorted(set(supp3a["target"]) - set(published_3a["target"])))

### ECM clustering and kNN neighborhoods

Leiden clustering on the matrisome genes, then k-means on the smoothed ECM-cluster composition
of each spot's grid neighbours. Run once to write `ecm_neighborhood_annotations_v4.csv`; with
`RUN_CLUSTERING = False` the frozen file is read instead.

In [ ]:
if RUN_CLUSTERING:
    # Leiden clustering based on matrisome
    # Parameters
    ecm_cluster_key   = "ECM_leiden"
    neighbors_key     = "ecm_neighbors"
    ecm_rep_key       = "X_ecm"
    umap_key          = "X_umap_ECM"

    leiden_resolution  = 0.6
    n_neighbors        = 20
    ecm_umap_min_dist  = 0.5
    ecm_umap_spread    = 1.2
    LAYER              = "log_norm_count"
    metric             = "euclidean"
    random_state       = 0

    adata_target = adata
    t_start = time.time()

    # 1) Select ECM genes present in this object
    ecm_genes_filtered = [g for g in ecm_genes_brain if g in adata_target.var_names]
    print(f"ECM genes: {len(ecm_genes_brain)} panel, {len(ecm_genes_filtered)} present")

    # 2) Extract matrix
    if LAYER is not None and LAYER in adata_target.layers:
        X_layer = adata_target.layers[LAYER]
    else:
        X_layer = adata_target.X

    idxs = adata_target.var_names.get_indexer(ecm_genes_filtered)
    X_ecm = X_layer[:, idxs]
    if sparse.issparse(X_ecm):
        X_ecm = X_ecm.toarray()
    else:
        X_ecm = np.asarray(X_ecm)

    # 3) Drop zero-variance genes
    gene_var = X_ecm.var(axis=0)
    keep = gene_var > 0
    if keep.sum() < len(ecm_genes_filtered):
        dropped = [g for g, k in zip(ecm_genes_filtered, keep) if not k]
        print(f"Dropping {len(dropped)} zero-variance genes: {dropped}")
        X_ecm = X_ecm[:, keep]
        ecm_genes_filtered = [g for g, k in zip(ecm_genes_filtered, keep) if k]

    # 4) Per-sample z-scoring
    samples = adata_target.obs["sample"].to_numpy()
    unique_samples, sample_inverse = np.unique(samples, return_inverse=True)

    X_ecm_z = np.zeros_like(X_ecm, dtype=np.float32)
    for i in range(len(unique_samples)):
        mask = (sample_inverse == i)
        X_s = X_ecm[mask]
        mu = X_s.mean(axis=0)
        sd = X_s.std(axis=0)
        sd[sd == 0] = 1.0
        X_ecm_z[mask] = (X_s - mu) / sd

    # 5) Post-z-score NaN check
    nan_mask = np.isnan(X_ecm_z).any(axis=0)
    if nan_mask.any():
        dropped = [g for g, k in zip(ecm_genes_filtered, nan_mask) if k]
        print(f"Dropping {len(dropped)} NaN genes: {dropped}")
        X_ecm_z = X_ecm_z[:, ~nan_mask]
        ecm_genes_filtered = [g for g, k in zip(ecm_genes_filtered, ~nan_mask) if k]

    # 6) Store ECM representation
    adata_target.obsm[ecm_rep_key] = X_ecm_z
    adata_target.uns["ecm_genes_used"] = ecm_genes_filtered
    print(f"ECM feature matrix shape: {X_ecm_z.shape}, elapsed: {time.time()-t_start:.0f}s")

    # 7) Neighbors
    t1 = time.time()
    print("Building neighbors...")
    sc.pp.neighbors(adata_target, use_rep=ecm_rep_key, n_neighbors=n_neighbors,
                    key_added=neighbors_key, metric=metric, random_state=random_state)
    print(f"  Neighbors done: {time.time()-t1:.0f}s")

    # 8) Leiden (igraph for speed on large datasets)
    t2 = time.time()
    print("Running Leiden (igraph)...")
    sc.tl.leiden(adata_target, resolution=leiden_resolution, key_added=ecm_cluster_key,
                 neighbors_key=neighbors_key, flavor="igraph", n_iterations=2)
    print(f"  Leiden done: {time.time()-t2:.0f}s")

    # 9) UMAP
    t3 = time.time()
    print("Running UMAP...")
    sc.tl.umap(adata_target, neighbors_key=neighbors_key, min_dist=ecm_umap_min_dist,
               spread=ecm_umap_spread, random_state=random_state)
    adata_target.obsm[umap_key] = adata_target.obsm["X_umap"]
    print(f"  UMAP done: {time.time()-t3:.0f}s")

    n_clust = adata_target.obs[ecm_cluster_key].nunique()
    print(f"\nTotal time: {time.time()-t_start:.0f}s, Clusters: {n_clust}")
    print(adata_target.obs[ecm_cluster_key].value_counts().sort_index())

In [ ]:
if RUN_CLUSTERING:
    # ### Save ECM clustering + UMAP to disk


    RESULTS_DIR    = OUT
    RESULTS_PREFIX = "ecm_neighborhoods_v4"  # change if you make new versions

    # pick AnnData
    if "adata_target" in locals():
        ad = adata_target
    elif "adata" in locals():
        ad = adata
    else:
        raise ValueError("No AnnData object found (adata_target or adata).")

    cluster_key = "ECM_leiden"
    umap_key    = "X_umap_ECM"
    ecm_rep_key = "X_ecm"
    neighbors_key = "ecm_neighbors"

    # sanity checks
    if cluster_key not in ad.obs.columns:
        raise KeyError(f"obs['{cluster_key}'] not found. Run ECM clustering first.")
    if umap_key not in ad.obsm:
        raise KeyError(f"obsm['{umap_key}'] not found. Run ECM UMAP first.")

    os.makedirs(RESULTS_DIR, exist_ok=True)

    # 1) Save per-spot cluster + UMAP as CSV
    df = pd.DataFrame(index=ad.obs_names)
    df[cluster_key] = ad.obs[cluster_key].astype(str)

    umap = np.asarray(ad.obsm[umap_key])
    if umap.shape[1] < 2:
        raise ValueError(f"{umap_key} has shape {umap.shape}, expected at least 2 columns.")

    df["ECM_UMAP1"] = umap[:, 0]
    df["ECM_UMAP2"] = umap[:, 1]

    csv_path = os.path.join(RESULTS_DIR, f"{RESULTS_PREFIX}.csv")
    df.to_csv(csv_path)
    print("Saved ECM clusters + UMAP to:", csv_path)

    # 2) Save metadata (ECM genes used, keys, etc.) as JSON
    meta = {
        "ecm_genes_used": list(ad.uns.get("ecm_genes_used", [])),
        "cluster_key": cluster_key,
        "umap_key": umap_key,
        "ecm_rep_key": ecm_rep_key,
        "neighbors_key": neighbors_key,
    }

    json_path = os.path.join(RESULTS_DIR, f"{RESULTS_PREFIX}_meta.json")
    with open(json_path, "w") as f:
        json.dump(meta, f, indent=2)

    print("Saved ECM metadata to:", json_path)

In [ ]:
if RUN_CLUSTERING:
    ### ECM neighborhood analysis (sample-balanced, self-aware, tunable)



    # USER SETTINGS
    cluster_key   = "ECM_leiden"        # categorical ECM clusters per spot
    nhood_key     = "ECM_neighborhood"  # output column name for neighborhoods
    coord_type    = "grid"              # Visium hex topology
    n_rings       = 1                   # spatial radius in rings (1 = immediate neighbors)

    # Smoothing of neighbor composition
    alpha_global  = 0.1                 # shrink toward uniform composition
    alpha_self    = 1.0                 # weight of the seed spot's own ECM cluster
    pseudocount   = 1e-6                # floor to avoid zeros

    apply_hellinger = True              # Hellinger transform of compositions

    # Embedding
    n_pca_components = 15               # PCA dimensions for neighborhood vectors

    # k-means clustering
    kmeans_n_clusters    = 10           # number of neighborhoods
    kmeans_n_init        = 25
    kmeans_random_state  = 42

    # Sample-balanced training (optional for speed)
    max_spots_per_sample_for_kmeans = None  # None to use all spots; otherwise cap per sample ex 2000

    # Reporting thresholds
    min_patients_report = 5             # neighborhoods present in fewer than this many patients are "rare"


    # 0) CHECK INPUT

    if "adata_target" in locals():
        adata = adata_target
    elif "adata" in locals():
        adata = adata
    else:
        raise ValueError("Expected 'adata_target' or 'adata' in memory.")

    req_cols = ["sample", "imagecol", "imagerow", cluster_key]
    missing = [c for c in req_cols if c not in adata.obs.columns]
    if missing:
        raise ValueError(f"Missing required obs columns: {missing}")

    if not isinstance(adata.obs[cluster_key].dtype, CategoricalDtype):
        adata.obs[cluster_key] = adata.obs[cluster_key].astype("category")

    cats = adata.obs[cluster_key].cat.categories
    C = cats.size
    if C < 2:
        raise ValueError(f"Need >= 2 ECM clusters in '{cluster_key}', found {C}.")

    print(f"Using AnnData with {adata.n_obs} spots, {C} ECM clusters.")
    print(f"Neighborhoods based on {coord_type} topology, n_rings = {n_rings}.")


    # 1–5) PER-SAMPLE NEIGHBOR COMPOSITIONS (with self + global smoothing)

    samples = adata.obs["sample"].unique().tolist()
    all_feats = []
    all_barcodes = []
    all_samples = []

    for sid in samples:
        ad_s = adata[adata.obs["sample"] == sid].copy()
        if ad_s.n_obs == 0:
            continue

        if "spatial" not in ad_s.obsm:
            ad_s.obsm["spatial"] = ad_s.obs[["imagecol", "imagerow"]].to_numpy()

        # Build spatial neighbors for this sample
        sq.gr.spatial_neighbors(ad_s, coord_type=coord_type, n_rings=n_rings)
        A = ad_s.obsp["spatial_connectivities"].tocsr()

        # One-hot of ECM cluster labels (N x C)
        codes = ad_s.obs[cluster_key].cat.codes.to_numpy()
        X_one = np.eye(C, dtype=np.float32)[codes]

        # Neighbor composition (counts)
        X_nb = A.dot(X_one).astype(np.float32)

        # Smoothing: self + global + floor
        # self: add alpha_self to the seed's own cluster
        # global: alpha_global / C added to each cluster as weak prior
        X_nb += alpha_self * X_one
        X_nb += alpha_global * (1.0 / C)
        X_nb += pseudocount

        # Normalize to probabilities
        row_sums = X_nb.sum(axis=1, dtype=np.float32)
        X_nb /= row_sums[:, None]

        all_feats.append(X_nb)
        all_barcodes.extend(ad_s.obs_names.to_list())
        all_samples.extend(ad_s.obs["sample"].astype(str).to_list())

        # Cleanup
        del ad_s, A, X_one, X_nb, row_sums, codes
        gc.collect()

    if not all_feats:
        raise RuntimeError("No neighborhood features were generated; check inputs and clusters.")


    # 6) COMBINE, OPTIONAL HELLINGER, PREP FOR PCA

    X = np.vstack(all_feats).astype(np.float32)   # (N_total x C)
    del all_feats
    gc.collect()

    all_barcodes = np.array(all_barcodes)
    all_samples_arr = np.array(all_samples)
    del all_samples
    gc.collect()

    if apply_hellinger:
        np.sqrt(X, out=X)

    print("Neighborhood feature matrix shape:", X.shape)


    # 7) SAMPLE-BALANCED SUBSET FOR PCA + K-MEANS

    rng = np.random.default_rng(kmeans_random_state)
    unique_samples = np.unique(all_samples_arr)

    train_indices = []

    if max_spots_per_sample_for_kmeans is None:
        train_indices = np.arange(X.shape[0], dtype=int)
    else:
        for sid in unique_samples:
            idx_s = np.where(all_samples_arr == sid)[0]
            if idx_s.size == 0:
                continue
            if idx_s.size > max_spots_per_sample_for_kmeans:
                idx_s = rng.choice(idx_s, size=max_spots_per_sample_for_kmeans, replace=False)
            train_indices.append(idx_s)
        if len(train_indices) == 0:
            raise RuntimeError("No training indices selected. Check sampling settings.")
        train_indices = np.concatenate(train_indices)

    train_indices = np.sort(train_indices)
    print(f"Training subset size for PCA + k-means: {train_indices.size} spots "
          f"({train_indices.size / X.shape[0]:.3%} of all spots).")

    X_train = X[train_indices]


    # 8) PCA ON TRAINING SUBSET, TRANSFORM ALL

    p = min(n_pca_components, X.shape[1] - 1)
    if p < 1:
        raise ValueError("Cannot compute PCA with fewer than 1 component.")

    pca = PCA(n_components=p, svd_solver="randomized", random_state=kmeans_random_state)
    X_train_pca = pca.fit_transform(X_train).astype(np.float32)
    X_pca = pca.transform(X).astype(np.float32)

    print(f"PCA components used: {p}")
    print("Explained variance (first 5 PCs):", np.round(pca.explained_variance_ratio_[:5], 4))

    del X_train
    gc.collect()


    # 9) K-MEANS ON TRAINING SUBSET, PREDICT ALL

    km = KMeans(
        n_clusters=kmeans_n_clusters,
        n_init=kmeans_n_init,
        random_state=kmeans_random_state,
        verbose=0,
    )

    km.fit(X_train_pca)
    labels_all = km.predict(X_pca).astype(np.int32)

    del X_pca, X_train_pca
    gc.collect()

    print("k-means clustering complete.")
    print("Neighborhood label range:", labels_all.min(), "to", labels_all.max())


    # 10) MAP BACK TO FULL OBJECT

    labels_ser = pd.Series(labels_all, index=pd.Index(all_barcodes, name="barcode"))
    labels_ser = labels_ser.astype("int32").astype("string")
    mapped = labels_ser.reindex(adata.obs_names)

    adata.obs[nhood_key] = mapped.astype("category")

    sizes = adata.obs[nhood_key].value_counts(dropna=True).sort_values(ascending=False)
    print(f"Number of ECM neighborhoods: {sizes.size}")
    print("\nTop neighborhood sizes:")
    print(sizes.head(20))


    # 11) PER-PATIENT COVERAGE SUMMARY

    if "sample" not in adata.obs.columns:
        raise KeyError("adata.obs['sample'] is required for per-patient summaries.")

    cluster_sizes = adata.obs[nhood_key].value_counts(dropna=True)
    patient_counts = (
        adata.obs.dropna(subset=[nhood_key])
             .groupby(nhood_key)["sample"]
             .nunique()
    )

    summary = pd.DataFrame({
        "n_spots": cluster_sizes,
        "n_patients": patient_counts,
    }).sort_values("n_spots", ascending=False)

    print("\nNeighborhood per-patient coverage summary (head):")
    print(summary.head(20))

    rare = summary[summary["n_patients"] < min_patients_report]
    print(f"\nNeighborhoods present in fewer than {min_patients_report} patients: {rare.shape[0]}")
    if not rare.empty:
        print(rare)

In [ ]:
if RUN_CLUSTERING:
    # path to save
    out_dir = OUT
    out_path = os.path.join(out_dir, "ecm_neighborhood_annotations_v4.csv")

    # UMAP for ECM neighborhoods
    umap_ecm = adata.obsm["X_umap_ECM"]

    # build table, indexed by spot id
    df_out = pd.DataFrame(
        {
            "sample": adata.obs["sample"].astype(str),
            "ECM_leiden": adata.obs["ECM_leiden"].astype(str),
            "ECM_neighborhood": adata.obs["ECM_neighborhood"].astype(str),
            "UMAP1_ECM": umap_ecm[:, 0],
            "UMAP2_ECM": umap_ecm[:, 1],
        },
        index=adata.obs_names,
    )
    df_out.index.name = "spot_id"

    df_out.to_csv(out_path)
    print(f"Saved ECM annotations for {df_out.shape[0]} spots to:\n{out_path}")

In [ ]:
# Reload ECM data
if not RUN_CLUSTERING:
    in_path = ECM_ANNOTATIONS_PATH

    df_ecm = pd.read_csv(in_path, index_col="spot_id")

    # align to current object; will error if names disagree, which is what you want
    df_ecm = df_ecm.loc[adata.obs_names]

    # add back to obs and obsm
    adata.obs["ECM_leiden"] = df_ecm["ECM_leiden"].astype("category")
    adata.obs["ECM_neighborhood"] = df_ecm["ECM_neighborhood"].astype("category")
    adata.obsm["X_umap_ECM"] = df_ecm[["UMAP1_ECM", "UMAP2_ECM"]].to_numpy()

    print("Merged ECM_leiden, ECM_neighborhood, and X_umap_ECM into adata.")

### Fig 4c: ECM neighborhoods on H&E

In [ ]:
# Cell 2: Plot ECM neighborhoods on H&E


# ===== USER CONFIG =====
sample_id = "CH_G_196a_vis"     # published panel (last run: "ER_G_666a_vis")
ecm_neighborhoods = None          # None for all, or [1, 2, 3] for specific
highlight_nhood = None            # int or None; if set, other nhoods are gray

figsize = (14, 12)
spot_alpha = 0.9
he_alpha = 0.5
margin_px = 800
max_view_px = 6000
spot_diam_um = 55.0
fallback_mpp = 0.5
title = None
title_fontsize = 14

show_legend = True                # True/False to toggle legend
show_title = True                 # True/False to toggle title

# ========================

S3_ALIAS = "pa-bi4jtrdsa151p4d3m-jepjxh9qrwgxphe8hd3r6exicahnoeuw1b-s3alias"
VISIUM_PREFIX = {
    "CH": "b5bd8840-cc6e-4de2-a78a-22315e9d1734",
    "ER": "40ce7e61-5ff6-467e-9972-0c85705f086c",
    "CR": "f0415978-2385-4067-b215-13e699a0f7a7",
}
ALIGNED_DIRNAME = "aligned_features_DinoV2ViTBaseDistilled_aquavit"

ECM_COLORS = {
    0: "#377EB8",   1: "#FF7F00",   2: "#4DAF4A",   3: "#E41A1C",   4: "#984EA3",
    5: "#8B4513",   6: "#F198B3",   7: "#808080",   8: "#9ACD32",   9: "#008B8B",
}
GRAY = "#C0C0C0"

# --- Load data ---
s3 = s3fs.S3FileSystem(anon=False)
center = sample_id[:2]
prefix = VISIUM_PREFIX[center]

def s3key(uri):
    return uri.replace("s3://", "")

coords_uri = f"s3://{S3_ALIAS}/{prefix}/v7/image_features/{ALIGNED_DIRNAME}/{sample_id}/filtered_coords.csv"
with s3.open(s3key(coords_uri), "rb") as f:
    coords_df = pd.read_csv(f)
coords_df["barcode"] = coords_df["barcode"].astype(str)

he_uri = f"s3://{S3_ALIAS}/{prefix}/v7/converted_he/{sample_id}.tif"
tmp = tempfile.NamedTemporaryFile(suffix=".tif", delete=False)
tmp.close()
s3.get(s3key(he_uri), tmp.name)
slide = openslide.OpenSlide(tmp.name)

ecm_df = pd.read_csv(ECM_ANNOTATIONS_PATH)
ecm_sample = ecm_df[ecm_df["sample"] == sample_id].copy()
ecm_sample["barcode"] = ecm_sample["spot_id"].str.replace(f"{sample_id}_", "", regex=False)
barcode_to_ecm = {}
for _, row in ecm_sample.iterrows():
    try:
        barcode_to_ecm[row["barcode"]] = int(float(row["ECM_neighborhood"]))
    except (ValueError, TypeError):
        continue

# --- Compute view region ---
x_full = coords_df["pxl_col_in_fullres"].values
y_full = coords_df["pxl_row_in_fullres"].values
barcodes = coords_df["barcode"].values

x0 = max(0, int(x_full.min() - margin_px))
y0 = max(0, int(y_full.min() - margin_px))
x1 = int(x_full.max() + margin_px)
y1 = int(y_full.max() + margin_px)
w, h = x1 - x0, y1 - y0

scale = max(w / max_view_px, h / max_view_px, 1.0)
level = slide.get_best_level_for_downsample(scale)
ds = slide.level_downsamples[level]

region = slide.read_region((x0, y0), level, (int(w / ds), int(h / ds))).convert("RGB")
img = np.asarray(region)

mpp = float(slide.properties.get("openslide.mpp-x", fallback_mpp))
r = (spot_diam_um / 2) / mpp / ds

x = (x_full - x0) / ds
y = (y_full - y0) / ds
n_spots = len(x)

# --- Build spot colors ---
if ecm_neighborhoods is None:
    ecm_neighborhoods = list(range(10))

rgba = np.zeros((n_spots, 4))
for i, bc in enumerate(barcodes):
    ecm = barcode_to_ecm.get(bc, -1)
    if ecm < 0:
        continue
    if highlight_nhood is not None:
        if ecm == highlight_nhood:
            rgba[i, :3] = to_rgb(ECM_COLORS[ecm])
            rgba[i, 3] = spot_alpha
        elif ecm in ecm_neighborhoods:
            rgba[i, :3] = to_rgb(GRAY)
            rgba[i, 3] = spot_alpha * 0.4
    else:
        if ecm in ecm_neighborhoods:
            rgba[i, :3] = to_rgb(ECM_COLORS[ecm])
            rgba[i, 3] = spot_alpha

# --- Plot ---
if show_legend:
    fig, (ax, ax_legend) = plt.subplots(1, 2, figsize=(figsize[0] + 3, figsize[1]),
                                         gridspec_kw={"width_ratios": [figsize[0], 2.5],
                                                      "wspace": 0.02})
else:
    fig, ax = plt.subplots(figsize=figsize)

ax.imshow(img, alpha=he_alpha)

patches = [Circle((x[i], y[i]), r) for i in range(n_spots) if rgba[i, 3] > 0]
rgba_visible = rgba[rgba[:, 3] > 0]
if len(patches) > 0:
    pc = PatchCollection(patches, linewidth=0)
    pc.set_facecolors(rgba_visible)
    ax.add_collection(pc)

ax.axis("off")
ax.grid(False)

if show_title:
    if title is None:
        if highlight_nhood is not None:
            auto_title = f"{sample_id} — ECM Neighborhood {highlight_nhood}"
        else:
            auto_title = f"{sample_id} — ECM Neighborhoods"
    else:
        auto_title = title
    ax.set_title(auto_title, fontsize=title_fontsize, fontweight="bold", pad=10)

# Legend in separate panel
if show_legend:
    if highlight_nhood is not None:
        legend_nhoods = [highlight_nhood]
    else:
        legend_nhoods = sorted(ecm_neighborhoods)

    handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=ECM_COLORS[n],
                      markersize=12, markeredgewidth=0, label=f"N{n}")
               for n in legend_nhoods]
    ax_legend.legend(handles=handles, loc="center left", frameon=False,
                     fontsize=11, title="ECM\nNeighborhood", title_fontsize=12,
                     handletextpad=0.5, labelspacing=0.6)
    ax_legend.axis("off")
    ax_legend.grid(False)

plt.tight_layout()

# --- Save ---
out_stem = f"{sample_id}_ECM" + (f"_N{highlight_nhood}" if highlight_nhood is not None else "")
plotted = pd.DataFrame({"barcode": barcodes, "ECM_neighborhood": [barcode_to_ecm.get(b, np.nan) for b in barcodes]})
output.save(OUT, out_stem, fig, plotted.set_index("barcode"), panel="Fig 4c")
plt.show()
slide.close()
os.remove(tmp.name)

### Fig 4d: heatmap for ECM panel

Figure 2 (top 40 "most variable" genes) is the published panel; the legend's "top 5" does not
describe it.

In [ ]:
# ### ECM gene expression per neighborhood — heatmap with dendrograms


# User config
nhood_key       = "ECM_neighborhood"
LAYER           = "log_norm_count"
CMAP            = "RdBu_r"
VMAX_PERCENTILE = 98
TOP_N_GENES     = 40             # for the focused-genes figure
SELECTION       = "variable"     # "variable" → top N by std across niches (bidirectional)
                                 # "per_niche" → top N per niche by |z|, union

# Filter to genes present in adata
ecm_genes_heatmap = [g for g in ecm_genes_brain if g in adata.var_names]
print(f"ECM genes available: {len(ecm_genes_heatmap)}")

# Mean expression per neighborhood
nhoods        = adata.obs[nhood_key].astype(str).values
valid         = ~pd.isna(adata.obs[nhood_key].values)
unique_nhoods = sorted([n for n in np.unique(nhoods[valid]) if n != "nan"],
                        key=lambda x: int(x))

gene_idx = adata.var_names.get_indexer(ecm_genes_heatmap)
X = adata.layers[LAYER][:, gene_idx]
if issparse(X):
    X = X.toarray()

nhood_means = {}
for nh in unique_nhoods:
    mask = (nhoods == nh) & valid
    nhood_means[nh] = X[mask].mean(axis=0)

mean_df = pd.DataFrame(nhood_means, index=ecm_genes_heatmap).T  # rows: niches, cols: genes
mean_z  = mean_df.apply(lambda x: (x - x.mean()) / (x.std() + 1e-8), axis=0)


def draw_heatmap(genes_to_plot, fig_path, fig_width,
                 gene_fontsize=11, label_fontsize=11, panel=None):
    """Clustered heatmap with aligned dendrograms for a subset of genes."""

    sub_z = mean_z[genes_to_plot]
    Z_r   = linkage(sub_z.values,    method="ward", metric="euclidean")
    Z_c   = linkage(sub_z.values.T,  method="ward", metric="euclidean")
    r_ord = leaves_list(Z_r)
    c_ord = leaves_list(Z_c)
    sub_z_ordered = sub_z.iloc[r_ord, c_ord]

    row_labels = [f"ECM_{sub_z.index[i]}" for i in r_ord]
    col_labels = [sub_z.columns[i]        for i in c_ord]

    vmax = np.percentile(np.abs(sub_z_ordered.values), VMAX_PERCENTILE)
    vmin = -vmax   # symmetric, diverging

    n_rows, n_cols = sub_z_ordered.shape
    fig_height = max(5.0, 0.35 * n_rows + 2.5)

    fig = plt.figure(figsize=(fig_width, fig_height))

    # Layout: [row labels | heatmap | row dendrogram]
    #         column dendrogram on top, aligned with heatmap column
    gs = fig.add_gridspec(
        2, 3,
        width_ratios=[2.0, 30, 1.5],
        height_ratios=[1.2, 6],
        hspace=0.02, wspace=0.02,
        left=0.04, right=0.95, top=0.95, bottom=0.20,
    )

    # Top-left corner: empty (above row labels)
    ax_corner = fig.add_subplot(gs[0, 0]); ax_corner.axis("off")

    # Top: column dendrogram (above the heatmap column)
    ax_dendro_col = fig.add_subplot(gs[0, 1])
    dendrogram(Z_c, no_labels=True, ax=ax_dendro_col,
               color_threshold=0, above_threshold_color="0.4")
    # scipy draws leaves at 5, 15, 25, ... (5 + 10*i). To align leaf centers
    # with imshow cell centers (0, 1, 2, ...), the dendrogram x-range needs
    # to span exactly the same physical width as the heatmap, with leaves
    # centered in cells. xlim 0 to 10*n_cols achieves this when both axes
    # have identical width via the gridspec.
    ax_dendro_col.set_xlim(0, n_cols * 10)
    ax_dendro_col.axis("off")

    # Top-right corner: empty (above row dendrogram)
    ax_corner_r = fig.add_subplot(gs[0, 2]); ax_corner_r.axis("off")

    # Left: row labels
    ax_labels = fig.add_subplot(gs[1, 0])
    ax_labels.set_xlim(0, 1)
    ax_labels.set_ylim(n_rows - 0.5, -0.5)
    for i, lbl in enumerate(row_labels):
        ax_labels.text(0.95, i, lbl, ha="right", va="center",
                       fontsize=label_fontsize)
    ax_labels.axis("off")

    # Heatmap (center)
    ax = fig.add_subplot(gs[1, 1])
    im = ax.imshow(sub_z_ordered.values, aspect="auto", cmap=CMAP,
                   vmin=vmin, vmax=vmax, interpolation="none")
    ax.set_xlim(-0.5, n_cols - 0.5)
    ax.set_ylim(n_rows - 0.5, -0.5)
    ax.set_yticks([])
    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(col_labels, rotation=45, fontsize=gene_fontsize,
                       ha="right", rotation_mode="anchor")
    ax.tick_params(axis="both", which="both", length=0)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)

    # Right: row dendrogram (leaves point left, toward the heatmap)
    ax_dendro_row = fig.add_subplot(gs[1, 2])
    dendrogram(Z_r, orientation="right", no_labels=True, ax=ax_dendro_row,
               color_threshold=0, above_threshold_color="0.4")
    # scipy draws leaves at 5, 15, ...; match imshow's row centering and
    # invert so row 0 is at the top
    ax_dendro_row.set_ylim(0, n_rows * 10)
    ax_dendro_row.invert_yaxis()
    ax_dendro_row.axis("off")

    # Colorbar — placed past the row dendrogram on the right
    cbar = plt.colorbar(im, ax=ax_dendro_row, orientation="vertical",
                        shrink=0.6, pad=0.4, fraction=0.4)
    cbar.set_label("Z-score", fontsize=10)
    cbar.ax.tick_params(labelsize=8)

    output.save(OUT, fig_path, fig, sub_z_ordered.rename(index=lambda i: f"ECM_{i}"), panel=panel)
    plt.show()
    print(f"Saved: {fig_path}  ({n_rows}×{n_cols})")
    return sub_z_ordered


# ---- Figure 1: full heatmap, all ECM genes ----
n_genes_full = len(ecm_genes_heatmap)
fig_w_full   = max(20.0, 0.32 * n_genes_full + 4.0)
print(f"\nFigure 1: full heatmap ({n_genes_full} genes, width={fig_w_full:.1f}\")")
draw_heatmap(
    genes_to_plot  = ecm_genes_heatmap,
    fig_path       = "ECM_neighborhood_heatmap_all",
    fig_width      = fig_w_full,
    gene_fontsize  = 12,
    label_fontsize = 13,
)

# ---- Figure 2: most-informative genes ----
if SELECTION == "variable":
    # Top N genes by standard deviation of z-score across niches.
    # These are the genes whose expression varies most between niches —
    # captures bidirectional signal (depletion AND enrichment).
    gene_var      = mean_z.std(axis=0)
    selected_genes = gene_var.sort_values(ascending=False).head(TOP_N_GENES).index.tolist()
    print(f"\nFigure 2: top {TOP_N_GENES} most-variable ECM genes (across niches)")
elif SELECTION == "per_niche":
    # Top N per niche by |z|, then union. Captures the strongest signed signal
    # per niche (enrichment or depletion) and combines.
    selected_genes = set()
    per_niche = max(3, TOP_N_GENES // len(unique_nhoods) + 2)
    for nh in unique_nhoods:
        top = mean_z.loc[nh].abs().sort_values(ascending=False).head(per_niche)
        selected_genes.update(top.index)
    selected_genes = list(selected_genes)
    print(f"\nFigure 2: top genes per niche (~{per_niche}/niche, "
          f"union → {len(selected_genes)} genes)")
else:
    raise ValueError(f"SELECTION must be 'variable' or 'per_niche', got {SELECTION!r}")

fig_w_top = max(12.0, 0.40 * len(selected_genes) + 4.0)
fig4d = draw_heatmap(
    genes_to_plot  = selected_genes,
    fig_path       = f"ECM_neighborhood_heatmap_top{len(selected_genes)}",
    fig_width      = fig_w_top,
    gene_fontsize  = 18,
    label_fontsize = 18,
    panel          = "Fig 4d",
)
print("rows:", [f"ECM_{i}" for i in fig4d.index])
print("columns:", list(fig4d.columns))
print("same genes and order as the published panel:", list(fig4d.columns) == PUBLISHED_FIG4D_GENES)

### ECM neighborhood gene signatures

In [ ]:
nhood_key = "ECM_neighborhood"
LAYER = "log_norm_count"
MIN_SPOTS_PER_PSEUDOBULK = 30   # min spots for a patient-neighborhood combo
MIN_PATIENTS_PER_NHOOD = 10     # neighborhood must appear in this many patients
TOP_N_FULL = 10
TOP_N_SHORT = 5
LOGFC_THRESH = 0.25
FDR_THRESH = 0.05

# Which gene sets to test
ECM_ONLY = True   # True = ECM panel only, False = full transcriptome

if ECM_ONLY:
    test_genes = [g for g in ecm_genes_brain if g in adata.var_names]
    label = "ECM-restricted"
else:
    test_genes = list(adata.var_names)
    label = "Full transcriptome"

print(f"Running DE on {len(test_genes)} genes ({label})")

# 1) Pseudobulk: mean expression per patient per neighborhood
samples = adata.obs["sample"].values
nhoods = adata.obs[nhood_key].astype(str).values

# Extract expression matrix for test genes
gene_idx = adata.var_names.get_indexer(test_genes)
X = adata.layers[LAYER][:, gene_idx]
if issparse(X):
    X = X.toarray()

# Build pseudobulk table
pb_records = []
for sample_id in np.unique(samples):
    for nh in np.unique(nhoods):
        mask = (samples == sample_id) & (nhoods == nh)
        n_spots = mask.sum()
        if n_spots < MIN_SPOTS_PER_PSEUDOBULK:
            continue
        mean_expr = X[mask].mean(axis=0)
        pb_records.append({
            "sample": sample_id,
            "neighborhood": nh,
            "n_spots": n_spots,
            "expr": mean_expr,
        })

pb_df = pd.DataFrame(pb_records)
print(f"Pseudobulk entries: {len(pb_df)}")
print(f"Patients per neighborhood:")
print(pb_df.groupby("neighborhood")["sample"].nunique().sort_values(ascending=False))

# Filter neighborhoods with enough patients
nh_counts = pb_df.groupby("neighborhood")["sample"].nunique()
valid_nhoods = nh_counts[nh_counts >= MIN_PATIENTS_PER_NHOOD].index.tolist()
pb_df = pb_df[pb_df["neighborhood"].isin(valid_nhoods)]
print(f"\nNeighborhoods passing patient filter: {len(valid_nhoods)}")

# 2) One-vs-rest Wilcoxon on pseudobulked values
expr_matrix = np.vstack(pb_df["expr"].values)  # (n_pseudobulks x n_genes)
neighborhoods = pb_df["neighborhood"].values

all_results = []

for nh in sorted(valid_nhoods):
    mask_in = neighborhoods == nh
    mask_out = ~mask_in
    
    n_in = mask_in.sum()
    n_out = mask_out.sum()
    
    if n_in < MIN_PATIENTS_PER_NHOOD:
        continue
    
    X_in = expr_matrix[mask_in]
    X_out = expr_matrix[mask_out]
    
    pvals = []
    logfcs = []
    
    for j in range(len(test_genes)):
        vals_in = X_in[:, j]
        vals_out = X_out[:, j]
        
        lfc = vals_in.mean() - vals_out.mean()
        
        try:
            stat, p = ranksums(vals_in, vals_out)
        except Exception:
            p = 1.0
        
        pvals.append(p)
        logfcs.append(lfc)
    
    pvals = np.array(pvals)
    logfcs = np.array(logfcs)
    
    # FDR correction
    reject, fdr, _, _ = multipletests(pvals, method="fdr_bh")
    
    de_df = pd.DataFrame({
        "gene": test_genes,
        "logFC": logfcs,
        "pval": pvals,
        "fdr": fdr,
        "mean_in": X_in.mean(axis=0),
        "mean_out": X_out.mean(axis=0),
    })
    
    # Filter and rank
    sig = de_df[(de_df["fdr"] < FDR_THRESH) & (de_df["logFC"] > LOGFC_THRESH)].copy()
    sig = sig.sort_values(["logFC"], ascending=False)
    
    top10 = sig.head(TOP_N_FULL)["gene"].tolist()
    top5 = sig.head(TOP_N_SHORT)["gene"].tolist()
    
    all_results.append({
        "neighborhood": nh,
        "n_pseudobulks": n_in,
        "n_sig_genes": len(sig),
        "top10": top10,
        "top5": top5,
        "de_full": de_df.sort_values("pval"),
    })
    
    print(f"\nNeighborhood {nh} (n={n_in} pseudobulks, {len(sig)} sig genes):")
    print(f"  Top 10: {', '.join(top10) if top10 else 'None pass filters'}")
    print(f"  Top 5:  {', '.join(top5) if top5 else 'None pass filters'}")

# 3) Summary table
print(f"\n{'='*90}")
print(f"ECM NEIGHBORHOOD SIGNATURES ({label})")
print(f"{'='*90}")
header = f"{'Nhood':<8} {'N_pb':>5} {'N_sig':>6} | {'Top 5 markers':<45} | {'Top 10 markers'}"
print(header)
print("-" * len(header))

for r in sorted(all_results, key=lambda x: x["neighborhood"]):
    t5 = ", ".join(r["top5"]) if r["top5"] else "---"
    t10 = ", ".join(r["top10"]) if r["top10"] else "---"
    print(f"{r['neighborhood']:<8} {r['n_pseudobulks']:>5} {r['n_sig_genes']:>6} | {t5:<45} | {t10}")

# 4) Check for shared genes across top-5 lists
all_top5 = [g for r in all_results for g in r["top5"]]
dupes = {g: c for g, c in Counter(all_top5).items() if c > 1}
print(f"\nShared genes across top-5 lists: {dupes if dupes else 'None'}")

# 5) Save full DE results
all_de = []
for r in all_results:
    df_de = r["de_full"].copy()
    df_de["neighborhood"] = r["neighborhood"]
    all_de.append(df_de)

output.save_table(OUT, f"ECM_neighborhood_signatures_{label.replace(' ', '_')}", pd.concat(all_de), index=False)
print(f"\nSaved full DE results")

### ECM gene signature using all genes

Pseudobulk means of every gene from per section × neighborhood sums
(`nbhd.pseudobulk_sums`, read a few sections at a time) instead of the full matrix.

In [ ]:
nhood_key = "ECM_neighborhood"
LAYER = "log_norm_count"
MIN_SPOTS_PER_PSEUDOBULK = 30
MIN_PATIENTS_PER_NHOOD = 10
TOP_N_FULL = 10
TOP_N_SHORT = 5
LOGFC_THRESH = 0.25
FDR_THRESH = 0.05

nhoods = adata.obs[nhood_key].astype(str)
pb_sums, pb_n = nbhd.pseudobulk_sums(nhoods, normalization=LAYER)
samples = adata.obs["sample"].values
nhoods = nhoods.values

test_genes = list(pb_sums.columns)
print(f"Running DE on {len(test_genes)} genes (full transcriptome)")

unique_samples = np.unique(samples)
unique_nhoods = np.unique(nhoods)

pb_list = []
pb_meta = []

for nh in unique_nhoods:
    for sid in unique_samples:
        n_spots = int(pb_n.get((sid, nh), 0))
        if n_spots < MIN_SPOTS_PER_PSEUDOBULK:
            continue
        mean_expr = pb_sums.loc[(sid, nh)].to_numpy() / n_spots
        
        pb_list.append(mean_expr)
        pb_meta.append({"sample": sid, "neighborhood": nh, "n_spots": n_spots})

pb_matrix = np.vstack(pb_list)
pb_df = pd.DataFrame(pb_meta)

print(f"Pseudobulk matrix: {pb_matrix.shape}")
print(f"Patients per neighborhood:")
print(pb_df.groupby("neighborhood")["sample"].nunique().sort_values(ascending=False))

nh_counts = pb_df.groupby("neighborhood")["sample"].nunique()
valid_nhoods = nh_counts[nh_counts >= MIN_PATIENTS_PER_NHOOD].index.tolist()
print(f"Neighborhoods passing filter: {len(valid_nhoods)}")

neighborhoods = pb_df["neighborhood"].values

all_results_full = []

for nh in sorted(valid_nhoods):
    mask_in = neighborhoods == nh
    mask_out = ~mask_in
    
    n_in = mask_in.sum()
    if n_in < MIN_PATIENTS_PER_NHOOD:
        continue
    
    X_in = pb_matrix[mask_in]
    X_out = pb_matrix[mask_out]
    
    mean_in = X_in.mean(axis=0)
    mean_out = X_out.mean(axis=0)
    logfcs = mean_in - mean_out
    
    pvals = np.ones(len(test_genes))
    for j in range(len(test_genes)):
        try:
            _, pvals[j] = ranksums(X_in[:, j], X_out[:, j])
        except Exception:
            pass
    
    _, fdr, _, _ = multipletests(pvals, method="fdr_bh")
    
    de_df = pd.DataFrame({
        "gene": test_genes,
        "logFC": logfcs,
        "pval": pvals,
        "fdr": fdr,
        "mean_in": mean_in,
        "mean_out": mean_out,
    })
    
    sig_up = de_df[(de_df["fdr"] < FDR_THRESH) & (de_df["logFC"] > LOGFC_THRESH)].copy()
    sig_up = sig_up.sort_values("logFC", ascending=False)
    
    top10 = sig_up.head(TOP_N_FULL)["gene"].tolist()
    top5 = sig_up.head(TOP_N_SHORT)["gene"].tolist()
    
    sig_down = de_df[(de_df["fdr"] < FDR_THRESH) & (de_df["logFC"] < -LOGFC_THRESH)].copy()
    sig_down = sig_down.sort_values("logFC", ascending=True)
    
    top5_down = sig_down.head(TOP_N_SHORT)["gene"].tolist()
    
    all_results_full.append({
        "neighborhood": nh,
        "n_pseudobulks": n_in,
        "n_sig_up": len(sig_up),
        "n_sig_down": len(sig_down),
        "top10": top10,
        "top5": top5,
        "top5_down": top5_down,
        "de_full": de_df.sort_values("pval"),
    })
    
    print(f"\nNeighborhood {nh} (n={n_in}, {len(sig_up)} up, {len(sig_down)} down):")
    print(f"  Top 5 UP:   {', '.join(top5) if top5 else '---'}")
    print(f"  Top 5 DOWN: {', '.join(top5_down) if top5_down else '---'}")
    print(f"  Top 10 UP:  {', '.join(top10) if top10 else '---'}")

print(f"\n{'='*100}")
print(f"FULL TRANSCRIPTOME SIGNATURES")
print(f"{'='*100}")
for r in sorted(all_results_full, key=lambda x: x["neighborhood"]):
    t5 = ", ".join(r["top5"]) if r["top5"] else "---"
    t5d = ", ".join(r["top5_down"]) if r["top5_down"] else "---"
    print(f"\nNhood {r['neighborhood']} ({r['n_pseudobulks']} pb, {r['n_sig_up']} up, {r['n_sig_down']} down):")
    print(f"  UP:   {t5}")
    print(f"  DOWN: {t5d}")

all_top5 = [g for r in all_results_full for g in r["top5"]]
dupes = {g: c for g, c in Counter(all_top5).items() if c > 1}
print(f"\nShared genes in top-5 UP: {dupes if dupes else 'None'}")

all_de = []
for r in all_results_full:
    df_de = r["de_full"].copy()
    df_de["neighborhood"] = r["neighborhood"]
    all_de.append(df_de)

output.save_table(OUT, "ECM_neighborhood_signatures_full_transcriptome", pd.concat(all_de), index=False)
print("Saved: ECM_neighborhood_signatures_full_transcriptome.csv")

In [ ]:
# Compare with the signature tables of the notebook's last run
for stem, res in [("ECM_neighborhood_signatures_ECM-restricted", all_results),
                  ("ECM_neighborhood_signatures_full_transcriptome", all_results_full)]:
    pub = pd.read_csv(PUBLISHED_NB_DIR / f"{stem}.csv")
    new = pd.concat([r["de_full"].assign(neighborhood=r["neighborhood"]) for r in res])
    m = new.astype({"neighborhood": str}).merge(pub.astype({"neighborhood": str}), on=["neighborhood", "gene"], suffixes=("", "_pub"))
    print(f"{stem}: {len(m)} of {len(pub)} published rows matched; max |Δ logFC| = {(m['logFC'] - m['logFC_pub']).abs().max():.2e}, "
          f"max |Δ p| = {(m['pval'] - m['pval_pub']).abs().max():.2e}")
del pb_sums
gc.collect()

### Supp Fig 4f: ECM neighborhood cell landscape

In [ ]:
nhood_key    = "ECM_neighborhood"
c2l_prefix   = "meanscell_abundance_w_sf_"
curation_col = "celltype_level4_curated"

top_n        = 12
figsize      = (16, 6)
bar_width    = 0.75
other_color  = "#ebecf0"
legend_font  = 8
legend_ncol  = 1

# Exclude patterns (case-insensitive)
EXCLUDE_PATTERNS = ["doublet", "other"]

if nhood_key not in adata.obs.columns:
    raise ValueError(f"{nhood_key} not found in adata.obs")
if "celltype_annotation_key" not in adata.uns:
    raise ValueError("adata.uns['celltype_annotation_key'] not found.")

celltype_key = adata.uns["celltype_annotation_key"]

counts_curated = nbhd.c2l_curated_counts(adata.obs, celltype_key, curation_col, EXCLUDE_PATTERNS)

# Normalize to fractions
row_sums = counts_curated.sum(axis=1).replace(0, np.nan)
frac = counts_curated.div(row_sums, axis=0).fillna(0.0)

# Mean fraction per neighborhood, re-normalize
nh = adata.obs[nhood_key].astype("category")
means = (
    frac.groupby(nh)
        .mean()
        .pipe(lambda df: df.div(df.sum(axis=1), axis=0))
        .fillna(0.0)
)

# Order neighborhoods by hierarchical clustering
H = np.sqrt(means.values.astype(np.float64))
Z = linkage(H, method="average", metric="euclidean")
order_idx = leaves_list(Z)
nhood_order = [means.index[i] for i in order_idx]

# Get top N cell types per neighborhood
top_map = {}
for nh_name, row in means.iterrows():
    top_map[nh_name] = row.sort_values(ascending=False).head(top_n)

# Union of all top types, sorted by global importance
union_types = sorted(set().union(*[s.index for s in top_map.values()]))
global_importance = pd.Series(0.0, index=union_types)
for s in top_map.values():
    global_importance.loc[s.index] += s.values
union_types_sorted = list(global_importance.sort_values(ascending=False).index)

# Color palette
tab20 = list(plt.get_cmap("tab20").colors) + list(plt.get_cmap("tab20b").colors)
palette = {ct: to_hex(tab20[i % len(tab20)]) for i, ct in enumerate(union_types_sorted)}

# Build stacked bar data
x = np.arange(len(nhood_order))
heights_by_type = {
    ct: np.array([float(top_map[nh].get(ct, 0.0)) for nh in nhood_order])
    for ct in union_types_sorted
}
other_heights = np.array(
    [max(0.0, 1.0 - float(top_map[nh].sum())) for nh in nhood_order]
)

# Plot
fig, ax = plt.subplots(figsize=figsize)
bottom = np.zeros(len(nhood_order), dtype=float)

for ct in union_types_sorted:
    h = heights_by_type[ct]
    if np.all(h == 0):
        continue
    ax.bar(x, h, bottom=bottom, width=bar_width, label=ct,
           color=palette[ct], edgecolor="white", linewidth=0.3)
    bottom += h

if not np.all(other_heights == 0):
    ax.bar(x, other_heights, bottom=bottom, width=bar_width, label="Other",
           color=other_color, edgecolor="white", linewidth=0.3)

ax.set_xticks(x)
ax.set_xticklabels([f"N{nh}" for nh in nhood_order], fontsize=11, fontweight="bold")
ax.set_ylabel("Proportion", fontsize=11)
ax.set_ylim(0, 1.0)
ax.set_xlim(-0.5, len(x) - 0.5)
ax.margins(x=0.01)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_title("Cell-type composition per ECM neighborhood", fontsize=13, fontweight="bold")

# Legend outside right, sorted by importance
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels,
          loc="center left", bbox_to_anchor=(1.01, 0.5),
          frameon=False, fontsize=legend_font, ncol=legend_ncol,
          title="Cell type", title_fontsize=9, handlelength=1.2,
          handleheight=1.0, labelspacing=0.4)

plt.tight_layout()
plotted = pd.DataFrame(heights_by_type, index=[f"N{nh}" for nh in nhood_order]).assign(Other=other_heights)
output.save(OUT, "ECM_neighborhood_cell_landscape", fig, plotted, panel="Supp Fig 4f")
plt.show()

### Fig 4g: ECM adjacency

In [ ]:
# Cell 2: Spatial adjacency / co-occurrence between ECM neighborhoods


nhood_key = "ECM_neighborhood"
coord_type = "grid"
n_rings = 1

# Build contact frequency matrix across all samples
unique_nhoods = sorted(adata.obs[nhood_key].dropna().astype(str).unique())
n_nh = len(unique_nhoods)
nh_to_idx = {nh: i for i, nh in enumerate(unique_nhoods)}

contact_counts = np.zeros((n_nh, n_nh), dtype=np.float64)
samples = adata.obs["sample"].unique()

print(f"Computing spatial adjacency across {len(samples)} samples...")

for i, sid in enumerate(samples):
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(samples)}")
    
    ad_s = adata[adata.obs["sample"] == sid].copy()
    if ad_s.n_obs < 50:
        continue
    
    # Build spatial neighbors
    if "spatial" not in ad_s.obsm:
        if {"imagecol", "imagerow"}.issubset(ad_s.obs.columns):
            ad_s.obsm["spatial"] = ad_s.obs[["imagecol", "imagerow"]].to_numpy()
        else:
            continue
    
    sq.gr.spatial_neighbors(ad_s, coord_type=coord_type, n_rings=n_rings)
    A = ad_s.obsp["spatial_connectivities"].tocsr()
    
    labels = ad_s.obs[nhood_key].astype(str).values
    
    # Count contacts
    rows, cols = A.nonzero()
    for r, c in zip(rows, cols):
        if r < c:  # each edge once
            nh_r = labels[r]
            nh_c = labels[c]
            if nh_r in nh_to_idx and nh_c in nh_to_idx:
                ir = nh_to_idx[nh_r]
                ic = nh_to_idx[nh_c]
                contact_counts[ir, ic] += 1
                contact_counts[ic, ir] += 1
    
    del ad_s, A
    gc.collect()

print("Done.")

# Normalize: observed / expected under random mixing
row_totals = contact_counts.sum(axis=1)
total = contact_counts.sum()
expected = np.outer(row_totals, row_totals) / (total + 1e-8)
enrichment = np.log2((contact_counts + 1) / (expected + 1))

# Make DataFrames
nh_labels = [f"N{nh}" for nh in unique_nhoods]
contact_df = pd.DataFrame(contact_counts, index=nh_labels, columns=nh_labels)
enrich_df = pd.DataFrame(enrichment, index=nh_labels, columns=nh_labels)

# Cluster order
Z = linkage(enrichment, method="ward", metric="euclidean")
order = leaves_list(Z)
ordered_labels = [nh_labels[i] for i in order]

enrich_ordered = enrich_df.loc[ordered_labels, ordered_labels]
contact_ordered = contact_df.loc[ordered_labels, ordered_labels]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: Contact frequency (raw counts, log-scaled)
ax = axes[0]
ax.grid(False)
log_counts = np.log10(contact_ordered.values + 1)
im0 = ax.imshow(log_counts, cmap="viridis", aspect="equal", interpolation="none")
ax.set_xticks(range(len(ordered_labels)))
ax.set_xticklabels(ordered_labels, rotation=45, ha="right", fontsize=10, fontweight="bold")
ax.set_yticks(range(len(ordered_labels)))
ax.set_yticklabels(ordered_labels, fontsize=10, fontweight="bold")
ax.set_title("Spatial contact frequency\n(log10 counts)", fontsize=12, fontweight="bold")
cbar0 = plt.colorbar(im0, ax=ax, shrink=0.8)
cbar0.set_label("log10(contacts + 1)", fontsize=9)

# Panel B: Enrichment (log2 observed/expected)
ax = axes[1]
ax.grid(False)
vmax = np.percentile(np.abs(enrich_ordered.values), 95)
im1 = ax.imshow(enrich_ordered.values, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                aspect="equal", interpolation="none")
ax.set_xticks(range(len(ordered_labels)))
ax.set_xticklabels(ordered_labels, rotation=45, ha="right", fontsize=10, fontweight="bold")
ax.set_yticks(range(len(ordered_labels)))
ax.set_yticklabels(ordered_labels, fontsize=10, fontweight="bold")
ax.set_title("Spatial co-occurrence enrichment\n(log2 observed/expected)", fontsize=12, fontweight="bold")
cbar1 = plt.colorbar(im1, ax=ax, shrink=0.8)
cbar1.set_label("log2(O/E)", fontsize=9)

plt.tight_layout()
output.save(OUT, "ECM_neighborhood_spatial_adjacency", fig, contact_ordered, panel="Fig 4g")
plt.show()

# Print enrichment table
print("\nSpatial co-occurrence enrichment (log2 O/E):")
print(enrich_ordered.round(2).to_string())

# Top co-occurring pairs
pairs = []
for i in range(n_nh):
    for j in range(i+1, n_nh):
        pairs.append({
            "nhood_A": unique_nhoods[i],
            "nhood_B": unique_nhoods[j],
            "contacts": int(contact_counts[i, j]),
            "log2_enrichment": enrichment[i, j],
        })

pairs_df = pd.DataFrame(pairs).sort_values("log2_enrichment", ascending=False)
print("\nTop 10 co-occurring pairs:")
print(pairs_df.head(10).round(3).to_string(index=False))
print("\nTop 10 avoidant pairs:")
print(pairs_df.tail(10).round(3).to_string(index=False))

output.save_table(OUT, "ECM_neighborhood_spatial_adjacency", pairs_df, index=False)

### Nhood adjacency stats

In [ ]:
nhood_key = "ECM_neighborhood"
coord_type = "grid"
n_rings = 1
N_PERMS = 1000
SEED = 42

unique_nhoods = sorted(adata.obs[nhood_key].dropna().astype(str).unique())
n_nh = len(unique_nhoods)
nh_to_idx = {nh: i for i, nh in enumerate(unique_nhoods)}
samples = adata.obs["sample"].unique()

rng = np.random.default_rng(SEED)

# Precompute spatial graphs and labels per sample
print(f"Precomputing spatial graphs for {len(samples)} samples...")
sample_data = []

for i, sid in enumerate(samples):
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(samples)}")
    
    ad_s = adata[adata.obs["sample"] == sid].copy()
    if ad_s.n_obs < 50:
        continue
    
    if "spatial" not in ad_s.obsm:
        if {"imagecol", "imagerow"}.issubset(ad_s.obs.columns):
            ad_s.obsm["spatial"] = ad_s.obs[["imagecol", "imagerow"]].to_numpy()
        else:
            continue
    
    sq.gr.spatial_neighbors(ad_s, coord_type=coord_type, n_rings=n_rings)
    A = ad_s.obsp["spatial_connectivities"].tocsr()
    labels = ad_s.obs[nhood_key].astype(str).values
    
    # Store edges (upper triangle only)
    rows, cols = A.nonzero()
    mask = rows < cols
    edges = np.column_stack([rows[mask], cols[mask]])
    
    sample_data.append({"labels": labels, "edges": edges})
    del ad_s, A
    gc.collect()

print(f"Stored graphs for {len(sample_data)} samples")


def compute_contacts(sample_data, permute=False, rng=None):
    counts = np.zeros((n_nh, n_nh), dtype=np.float64)
    for sd in sample_data:
        labels = sd["labels"].copy()
        if permute:
            rng.shuffle(labels)
        nbhd.contact_counts(labels, sd["edges"], nh_to_idx, n_nh, out=counts)
    return counts


# Observed
print("Computing observed contacts...")
obs_counts = compute_contacts(sample_data, permute=False)
obs_totals = obs_counts.sum(axis=1)
obs_total = obs_counts.sum()
obs_expected = np.outer(obs_totals, obs_totals) / (obs_total + 1e-8)
obs_enrichment = np.log2((obs_counts + 1) / (obs_expected + 1))

# Permutations
print(f"Running {N_PERMS} permutations...")
perm_enrichments = np.zeros((N_PERMS, n_nh, n_nh))

for p in range(N_PERMS):
    if (p + 1) % 100 == 0:
        print(f"  Permutation {p+1}/{N_PERMS}")
    
    perm_counts = compute_contacts(sample_data, permute=True, rng=rng)
    perm_totals = perm_counts.sum(axis=1)
    perm_total = perm_counts.sum()
    perm_expected = np.outer(perm_totals, perm_totals) / (perm_total + 1e-8)
    perm_enrichments[p] = np.log2((perm_counts + 1) / (perm_expected + 1))

# Two-sided p-values
p_values = np.zeros((n_nh, n_nh))
for i in range(n_nh):
    for j in range(n_nh):
        obs_val = obs_enrichment[i, j]
        perm_vals = perm_enrichments[:, i, j]
        if obs_val >= 0:
            p_values[i, j] = (np.sum(perm_vals >= obs_val) + 1) / (N_PERMS + 1)
        else:
            p_values[i, j] = (np.sum(perm_vals <= obs_val) + 1) / (N_PERMS + 1)

# DataFrames
nh_labels = [f"N{nh}" for nh in unique_nhoods]
enrich_df = pd.DataFrame(obs_enrichment, index=nh_labels, columns=nh_labels)
pval_df = pd.DataFrame(p_values, index=nh_labels, columns=nh_labels)

print("\nEnrichment (log2 O/E):")
print(enrich_df.round(2).to_string())
print("\nP-values:")
print(pval_df.round(4).to_string())

# Significant pairs
pairs = []
for i in range(n_nh):
    for j in range(i+1, n_nh):
        pairs.append({
            "nhood_A": unique_nhoods[i],
            "nhood_B": unique_nhoods[j],
            "log2_enrichment": obs_enrichment[i, j],
            "p_value": p_values[i, j],
            "contacts": int(obs_counts[i, j]),
        })

pairs_df = pd.DataFrame(pairs).sort_values("p_value")
pairs_df["significant"] = pairs_df["p_value"] < 0.05

print(f"\nSignificant co-occurring (enriched, p<0.05):")
sig_pos = pairs_df[(pairs_df["significant"]) & (pairs_df["log2_enrichment"] > 0)]
print(sig_pos.round(4).to_string(index=False))

print(f"\nSignificant avoidant (depleted, p<0.05):")
sig_neg = pairs_df[(pairs_df["significant"]) & (pairs_df["log2_enrichment"] < 0)]
print(sig_neg.round(4).to_string(index=False))

output.save_table(OUT, "ECM_neighborhood_spatial_adjacency_with_pvals", pairs_df, index=False)

In [ ]:
# Compare with the tables of the notebook's last run
for stem, cols in [("ECM_neighborhood_spatial_adjacency", ["contacts", "log2_enrichment"]),
                   ("ECM_neighborhood_spatial_adjacency_with_pvals", ["contacts", "log2_enrichment", "p_value"])]:
    new = pd.read_csv(OUT / f"{stem}.csv", dtype={"nhood_A": str, "nhood_B": str})
    pub = pd.read_csv(PUBLISHED_NB_DIR / f"{stem}.csv", dtype={"nhood_A": str, "nhood_B": str})
    m = new.merge(pub, on=["nhood_A", "nhood_B"], suffixes=("", "_pub"))
    print(stem, f"{len(m)} pairs;", {c: float((m[c] - m[f"{c}_pub"]).abs().max()) for c in cols})

### ECM neighborhood abundance vs overall survival

In [ ]:
# Clinical tables of the three centres, all rows (as the notebook loaded them)
clin_out = cohort.clinical_as_ran().reset_index(drop=True)
print("Total rows:", len(clin_out))
print("Totals by center:\n", clin_out.groupby("center").size())

In [ ]:
# Build clin_dedup with join_key (one row per patient, prefer 'a' block)
clin_out["block_suffix"] = clin_out["Block sequence number per subject for MOSAIC"].astype(str)
clin_out["join_key"] = clin_out["Subject Id"].astype(str) + clin_out["block_suffix"]

suffix_order = {"a": 0, "b": 1, "c": 2}
clin_out["_sort"] = clin_out["block_suffix"].map(suffix_order).fillna(99)
clin_dedup = (
    clin_out.sort_values("_sort")
    .drop_duplicates(subset=["Subject Id"], keep="first")
    .drop(columns=["_sort"])
    .copy()
)
print(f"clin_dedup: {len(clin_dedup)} patients, {clin_dedup['join_key'].nunique()} unique join keys")

In [ ]:
nhood_key  = "ECM_neighborhood"
sample_key = "sample"
min_spots  = 100
top_labels = 10
cmap_name  = "tab20"

# Basic checks
for c in (nhood_key, sample_key):
    if c not in adata.obs.columns:
        raise KeyError(f"{c} not found in adata.obs")

sdf = adata.obs[[sample_key, nhood_key]].dropna().copy()
sdf[nhood_key] = sdf[nhood_key].astype("category")
if sdf[nhood_key].cat.categories.size == 0:
    raise ValueError("No neighborhoods present")

# Per-sample neighborhood fractions
counts = sdf.groupby([sample_key, nhood_key]).size().unstack(fill_value=0)
counts = counts.loc[counts.sum(axis=1) >= min_spots]
if counts.empty:
    raise ValueError("No samples pass min_spots filter")

fracs = counts.div(counts.sum(axis=1), axis=0)

# Link to clinical via join_key
map_df = pd.DataFrame({sample_key: fracs.index.astype(str)})
map_df["join_key"] = map_df[sample_key].str.replace(r"_vis$", "", regex=True).str[:9]

clin_os = clin_dedup[["join_key", "os_years"]].dropna(subset=["os_years"]).copy()

M = (
    fracs.reset_index()
         .merge(map_df, on=sample_key, how="left")
         .merge(clin_os, on="join_key", how="inner")
         .dropna(subset=["os_years"])
)

if M.empty:
    raise ValueError("No overlap between neighborhood fractions and clinical data")

print(f"Visium samples with neighborhoods: {fracs.shape[0]}")
print(f"Samples with OS after merge:       {M[sample_key].nunique()}")

# Spearman correlation per neighborhood
res = []
y = M["os_years"].to_numpy()
for c in fracs.columns:
    x = M[c].to_numpy()
    r, p = spearmanr(x, y)
    if np.isfinite(r) and np.isfinite(p):
        res.append({"neighborhood": str(c), "rho": r, "pval": p})

if not res:
    raise ValueError("No valid Spearman correlations")

df_res = pd.DataFrame(res)

# BH FDR
p = df_res["pval"].values
n = len(p)
order = np.argsort(p)
rank = np.arange(1, n + 1, dtype=float)
fdr = np.empty(n)
fdr[order] = np.minimum.accumulate((p[order] * n / rank)[::-1])[::-1]
df_res["fdr"] = np.clip(fdr, 0, 1)
df_res["neglog10p"] = -np.log10(np.clip(df_res["pval"].values, np.finfo(float).tiny, 1))
df_res = df_res.sort_values("pval").reset_index(drop=True)

# Color map consistent with other plots
cats = adata.obs[nhood_key].astype("category").cat.categories.astype(str).tolist()
cats_key = f"{nhood_key}_categories"
colors_key = f"{nhood_key}_colors"

if cats_key in adata.uns and colors_key in adata.uns:
    cm = dict(zip(adata.uns[cats_key], adata.uns[colors_key]))
else:
    cmap = plt.get_cmap(cmap_name, len(cats))
    colors = [to_hex(cmap(i)) for i in range(len(cats))]
    adata.uns[cats_key] = cats
    adata.uns[colors_key] = colors
    cm = dict(zip(cats, colors))

# Volcano plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    df_res["rho"], df_res["neglog10p"], s=90,
    c=[cm.get(k, "#666666") for k in df_res["neighborhood"]],
    edgecolors="white", linewidths=0.5,
)
ax.axvline(0, ls="--", c="grey", lw=1)
ax.axhline(-np.log10(0.05), ls="--", c="grey", lw=1)

for i in range(min(top_labels, len(df_res))):
    ax.annotate(
        df_res.at[i, "neighborhood"],
        (df_res.at[i, "rho"], df_res.at[i, "neglog10p"]),
        xytext=(0, 6), textcoords="offset points", ha="center", va="bottom", fontsize=10,
    )

ax.set(xlabel="Spearman ρ (neighborhood burden vs OS)", ylabel="-log10(p)",
       title="ECM neighborhood abundance vs overall survival")
ax.grid(False)
plt.tight_layout()
output.save(OUT, "ECM_neighborhood_vs_OS_spearman", fig, df_res.set_index("neighborhood"))
plt.show()

# Print and compare with the notebook's last run
print(df_res.to_string(index=False))
pub = pd.read_csv(PUBLISHED_NB_DIR / "ECM_neighborhood_vs_OS_spearman.csv", dtype={"neighborhood": str})
m = df_res.merge(pub, on="neighborhood", suffixes=("", "_pub"))
print(f"vs last run: max |Δ rho| = {(m['rho'] - m['rho_pub']).abs().max():.2e}, max |Δ p| = {(m['pval'] - m['pval_pub']).abs().max():.2e}")

In [ ]:
output.versions(["anndata", "scanpy", "squidpy", "scikit-learn", "scipy", "statsmodels", "openslide-python"])